# 02 · ทดสอบความเป็นไปได้ของ open-weight LLM บน Colab (ระยะที่ 3)

วัดสำหรับ **หนึ่งโมเดลต่อการรันหนึ่งครั้ง** (เปลี่ยน `MODEL_KEY` แล้ว *Runtime → Restart session* ก่อนรันโมเดลถัดไป):
- เวลาโหลด, หน่วยความจำ GPU, เวลาต่อ batch 40 prompts
- อัตราคำตอบที่ผ่าน schema (ครั้งแรก / หลัง retry) และชนิด error
- พฤติกรรมเปลี่ยนตามข้อมูลหรือไม่ (มีอาการ, ผู้ป่วยมากขึ้น → อยู่บ้านมากขึ้น?)
- ความสม่ำเสมอเมื่อ prompt เดิม (temperature 0 และ 0.7)

**ก่อนเริ่ม:** Runtime → Change runtime type → **T4 GPU**. ถ้าใช้ `llama8b` ต้องยอมรับ licence บน Hugging Face และใส่ `HF_TOKEN` ใน Colab Secrets

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
# Hugging Face token: needed only for gated models (llama8b). Add it in Colab: key icon (Secrets) -> HF_TOKEN
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded")
    except Exception:
        print("No HF_TOKEN secret: gated models will fail to download")

In [ ]:
MODEL_KEY = "qwen7b"        # @param ["typhoon8b", "qwen7b", "llama8b", "mistral7b", "qwen1_5b"]
BACKEND = cfg["llm"]["backend"]   # "transformers" (default) | "vllm" | "mock" (no GPU, pipeline test only)
N_SCENARIOS = 100

In [ ]:
import json, time, itertools
import numpy as np, pandas as pd
from gabm.agents import persona_text
from gabm.llm import build_resident_prompt, decide_batch, load_backend, parse_choice
from gabm.io import gpu_memory_used_mb, environment_info

personas = pd.read_csv(ROOT / cfg["personas"]["file"])
rng = np.random.default_rng(42)
HEALTH = {"healthy": "You feel healthy and have no symptoms.", "sick": "You have had fever and cough for 2 day(s)."}
CASES = [0.5, 5, 20, 60, 150]          # reported cases per 100k per day
scen = []
for j in range(N_SCENARIOS):
    p = personas.iloc[rng.integers(len(personas))]
    h = ["healthy", "sick"][int(rng.random() < 0.3)]
    c = CASES[j % len(CASES)]
    report = {"today_per100k": c, "avg7_per100k": round(c * 0.9, 1), "trend": "rising"}
    scen.append({"scenario": j, "agent_id": int(p.agent_id), "group": p.group, "health": h, "cases": c,
                 "prompt": build_resident_prompt(persona_text(p), 10, HEALTH[h], report, "None. Normal activities are allowed.", cfg["actions"])})
scen = pd.DataFrame(scen)
print(scen.prompt.iloc[0])

In [ ]:
mem0 = gpu_memory_used_mb()
t0 = time.time()
backend = load_backend(cfg["llm"], MODEL_KEY, BACKEND)
load_s = time.time() - t0
mem_load = gpu_memory_used_mb()
print(f"loaded {backend.name} in {load_s:.0f}s; GPU memory used {mem_load} MB")
print(backend.generate([scen.prompt.iloc[0]], temperature=0.0)[0])

## 1. ความเร็ว: หนึ่ง batch = หนึ่งวันของ 40 agents

In [ ]:
batch = scen.prompt.iloc[:cfg["sim"]["n_agents"]].tolist()
t0 = time.time(); outs = backend.generate(batch, temperature=cfg["llm"]["temperature"], seed=1, max_new_tokens=cfg["llm"]["max_new_tokens"])
batch_s = time.time() - t0
tok = getattr(backend, "tok", None)
n_tok = sum(len(tok(o)["input_ids"]) for o in outs) if tok else sum(len(o.split()) for o in outs)
mem_peak = gpu_memory_used_mb()
days, seeds = cfg["sim"]["days"], len(cfg["sim"]["seeds"])
print(f"{len(batch)} prompts in {batch_s:.1f}s  ({n_tok / batch_s:.0f} generated tokens/s), GPU memory {mem_peak} MB")
print(f"-> one {days}-day run ≈ {batch_s * days / 60:.1f} min; {seeds} seeds ≈ {batch_s * days * seeds / 60:.0f} min (without retries)")

## 2. Schema validity บน scenario ทั้งหมด

In [ ]:
fallbacks = ["stay_home"] * len(scen)
recs, sec = decide_batch(backend, scen.prompt.tolist(), list(cfg["actions"]), fallbacks, cfg["llm"], seed=7)
res = pd.concat([scen.drop(columns="prompt"), pd.DataFrame(recs)], axis=1)
valid_first = 1 - res.retried.mean(); valid_final = res.valid.mean()
print(f"valid on first try {valid_first:.1%}, after retry {valid_final:.1%}, fallback {res.fallback.mean():.1%}  ({sec:.0f}s)")
display(res.error.value_counts(dropna=False))
display(res[~res.valid | res.retried][["raw", "raw_retry"]].head())

## 3. พฤติกรรมเปลี่ยนตามข้อมูลหรือไม่ (หัวข้อ 9.3 ข้อ 2)

ถ้าสัดส่วน `stay_home` ไม่เปลี่ยนเลยตามจำนวนผู้ป่วยหรืออาการ โมเดลอาจไม่ได้ใช้ข้อมูลใน prompt

In [ ]:
ok = res[res.valid]
display(pd.crosstab([ok.health, ok.cases], ok.choice, normalize="index").round(2))
display(pd.crosstab(ok.group, ok.choice, normalize="index").round(2))

## 4. ความสม่ำเสมอ (consistency)

In [ ]:
sub = scen.iloc[:20]
reps = []
for r in range(5):
    out = backend.generate(sub.prompt.tolist(), temperature=cfg["llm"]["temperature"], seed=1000 + r, max_new_tokens=cfg["llm"]["max_new_tokens"])
    reps.append([parse_choice(o, list(cfg["actions"]))[0] for o in out])
reps = pd.DataFrame(reps).T
modal_agreement = reps.apply(lambda row: row.value_counts().iloc[0] / len(row) if row.notna().any() else np.nan, axis=1).mean()

g0 = [backend.generate(sub.prompt.tolist(), temperature=0.0, max_new_tokens=cfg["llm"]["max_new_tokens"]) for _ in range(2)]
identical_t0 = np.mean([a == b for a, b in zip(*g0)])
print(f"temperature {cfg['llm']['temperature']}: mean agreement with modal action over 5 repeats = {modal_agreement:.2f}")
print(f"temperature 0: identical raw output on repeat = {identical_t0:.0%}")

In [ ]:
FEAS = ROOT / "outputs" / "feasibility"; FEAS.mkdir(parents=True, exist_ok=True)
row = {"model_key": MODEL_KEY, "model_id": backend.name, "backend": BACKEND, "load_s": load_s,
       "gpu_mem_after_load_mb": mem_load, "gpu_mem_peak_mb": mem_peak, "batch_size": len(batch), "batch_s": batch_s,
       "tokens_per_s": n_tok / batch_s, "valid_first_try": valid_first, "valid_after_retry": valid_final,
       "modal_agreement_t": modal_agreement, "identical_t0": identical_t0,
       "est_run_min": batch_s * days / 60, "gpu": environment_info().get("gpu"), "date": pd.Timestamp.now().isoformat()}
f = FEAS / "feasibility.csv"
pd.DataFrame([row]).to_csv(f, mode="a", header=not f.exists(), index=False)
res.to_json(FEAS / f"scenarios_{MODEL_KEY}.jsonl", orient="records", lines=True, force_ascii=False)
display(pd.read_csv(f))
assert mem_peak is None or mem_peak < 15000, "peak GPU memory above 15 GB: reduce batch_size or use a smaller model"

In [ ]:
backend.unload()   # then Runtime -> Restart session, change MODEL_KEY, and run again for the next model